# LBF checkpoint evaluation

Evaluates Jumanji LBF checkpoints from `not_for_submit/checkpoints_for_eval_LBF`. The scenario axis is `env_id`, and the metric is `success_rate`.

In [5]:
import importlib
import os
import sys
from pathlib import Path

MPLCONFIGDIR = Path("/tmp") / "pci_marl_matplotlib"
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)
os.environ["MPLCONFIGDIR"] = str(MPLCONFIGDIR)

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "main.py").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root.")


ROOT = resolve_repo_root()
FIGURES_DIR = ROOT / "figures"
CHECKPOINTS_DIR = ROOT / "not_for_submit" / "checkpoints_for_eval_LBF"

for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

command_buffer_flag = "--xla_gpu_enable_command_buffer="
existing_xla_flags = os.environ.get("XLA_FLAGS", "")
if command_buffer_flag not in existing_xla_flags:
    os.environ["XLA_FLAGS"] = f"{existing_xla_flags} {command_buffer_flag}".strip()
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import sh_lbf_checkpoint_eval_utils as clu
clu = importlib.reload(clu)

DEVICE = "cuda"
CHECKPOINT_CHOICE = "final"  # or "best"
DUPLICATE_POLICY = "latest"
EVAL_SEEDS = list(range(10001, 10065))
EVAL_EPISODES = 1
EVAL_SEED_BATCH_SIZE = 16
TARGET_ENV_IDS = None  # e.g. ["2s-10x10-3p-3f"] or None for all discovered env_id values
ENV_RESULTS = {}

clu.bootstrap_project(DEVICE)
runs_df = clu.discover_lbf_runs(CHECKPOINTS_DIR, env_id_filter=TARGET_ENV_IDS)
if runs_df.empty:
    raise RuntimeError(f"No LBF runs were discovered under {CHECKPOINTS_DIR}")

In [6]:
TARGET_ENV_IDS = sorted(runs_df["scenario"].dropna().unique().tolist()) if TARGET_ENV_IDS is None else TARGET_ENV_IDS
TARGET_MODELS = clu.ordered_model_labels(runs_df["model_label"].dropna().unique().tolist(), clu.LBF_MODEL_ORDER)

print(f"Repo root: {ROOT}")
print(f"Loaded {len(runs_df)} LBF evaluation runs from {CHECKPOINTS_DIR}")
print(f"Env IDs: {TARGET_ENV_IDS}")
print(f"Models: {TARGET_MODELS}")
display(runs_df.groupby(["scenario", "model_label"]).size().unstack(fill_value=0))
display(runs_df[["scenario", "model_label", "seed", "wandb_run_name", "final_checkpoint"]].head())

Repo root: /home/chamjin/pci_marl
Loaded 79 LBF evaluation runs from /home/chamjin/pci_marl/not_for_submit/checkpoints_for_eval_LBF
Env IDs: ['2s-10x10-3p-3f', '2s-15x15-4p-3f', '2s-15x15-4p-5f', '2s-8x8-2p-2f-coop']
Models: ['ippo', 'ipci', 'mappo', 'mpci']


model_label,ipci,ippo,mappo,mpci
scenario,,,,
2s-10x10-3p-3f,5,5,5,5
2s-15x15-4p-3f,5,5,5,5
2s-15x15-4p-5f,5,5,5,5
2s-8x8-2p-2f-coop,5,4,5,5


,scenario,model_label,seed,wandb_run_name,final_checkpoint
0,2s-10x10-3p-3f,ipci,1,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...
1,2s-10x10-3p-3f,ipci,2,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...
2,2s-10x10-3p-3f,ipci,31,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...
3,2s-10x10-3p-3f,ipci,32,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...
4,2s-10x10-3p-3f,ipci,42,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...


In [7]:
# Smoke test: one training seed x two evaluation seeds.
SMOKE_EVAL_SEEDS = EVAL_SEEDS[:2]
smoke_row = runs_df.sort_values(["scenario", "model_label", "seed", "run_dir"]).iloc[0]
smoke_runs = runs_df[
    (runs_df["scenario"] == smoke_row["scenario"])
    & (runs_df["model_label"] == smoke_row["model_label"])
    & (runs_df["seed"] == int(smoke_row["seed"]))
].copy()
display(smoke_runs[["scenario", "model_label", "seed", "wandb_run_name", "run_dir"]])

smoke_result = clu.evaluate_map_constraint(
    smoke_runs,
    smoke_row["scenario"],
    "default",
    checkpoint_choice=CHECKPOINT_CHOICE,
    device=DEVICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=SMOKE_EVAL_SEEDS,
    eval_seed_batch_size=min(EVAL_SEED_BATCH_SIZE, len(SMOKE_EVAL_SEEDS)),
    duplicate_policy=DUPLICATE_POLICY,
)
display(smoke_result["run_results"])
display(smoke_result["seed_summary"])
display(smoke_result["model_summary"])

,scenario,model_label,seed,wandb_run_name,run_dir
0,2s-10x10-3p-3f,ipci,1,JumanjiLBF-2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...


/nodecache/users/chamjin/tmp/ipykernel_1810608/1756130118.py:11: UserWarning: std_success_rate is defined over training seeds. Filled 0.0 for rows with n_seeds=1.
      map_name constraint_key model_label  n_seeds
2s-10x10-3p-3f        default        ipci        1
  smoke_result = clu.evaluate_map_constraint(


,algorithm_type,checkpoint_choice,checkpoint_path,constraint_key,constraint_label,eval_max_reward,eval_mean_length,eval_mean_reward,eval_min_reward,eval_seed,...,map_name,metadata_path,model_key,model_label,num_episodes,project,run_dir,seed,train_seed,wandb_run_name
0,jax_pci_ippo,final,/home/chamjin/pci_marl/not_for_submit/checkpoi...,default,default,0.511111,38.0,0.333333,0.088889,10001,...,2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...,ipci,ipci,1,pci-marl-lbf,/home/chamjin/pci_marl/not_for_submit/checkpoi...,1,1,JumanjiLBF-2s-10x10-3p-3f
1,jax_pci_ippo,final,/home/chamjin/pci_marl/not_for_submit/checkpoi...,default,default,0.561905,19.0,0.333333,0.209524,10002,...,2s-10x10-3p-3f,/home/chamjin/pci_marl/not_for_submit/checkpoi...,ipci,ipci,1,pci-marl-lbf,/home/chamjin/pci_marl/not_for_submit/checkpoi...,1,1,JumanjiLBF-2s-10x10-3p-3f


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s-10x10-3p-3f,default,default,ipci,1,1.0,0.333333,28.5,1,2


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s-10x10-3p-3f,default,default,ipci,1.0,0.0,0.333333,NaN,28.5,NaN,1,1,2


## Full evaluation

Each checkpoint is evaluated with 64 individual evaluation seeds and one episode per seed.

In [8]:
ENV_RESULTS = {}
for env_id in TARGET_ENV_IDS:
    result = clu.evaluate_map(
        runs_df,
        env_id,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
        eval_seeds=EVAL_SEEDS,
        eval_seed_batch_size=EVAL_SEED_BATCH_SIZE,
        model_order=TARGET_MODELS,
        duplicate_policy=DUPLICATE_POLICY,
    )
    ENV_RESULTS[env_id] = result
    print(f"\n=== {env_id} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    latex = result["latex"].replace("Deterministic SMAX win rate", "Deterministic LBF success rate")
    latex = latex.replace("\\label{tab:smax_", "\\label{tab:lbf_")
    print(latex)

/home/chamjin/miniconda3/envs/pcimarl/lib/python3.10/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78


/home/chamjin/miniconda3/envs/pcimarl/lib/python3.10/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(
/home/chamjin/miniconda3/envs/pcimarl/lib/python3.10/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78


/home/chamjin/miniconda3/envs/pcimarl/lib/python3.10/site-packages/jax/_src/ops/scatter.py:92: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int32 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(



=== 2s-10x10-3p-3f ===


... (8 outputs left out)

\begin{table}[t]
\centering
\caption{Deterministic LBF success rate on 2s-15x15-4p-5f using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:lbf_2s-15x15-4p-5f_final}
\begin{tabular}{lcccc}
\toprule
Constraint & ippo & ipci & mappo & mpci \\
\midrule
default & 31.2 \scriptsize $\pm$ 8.2 & 36.6 \scriptsize $\pm$ 4.6 & 24.1 \scriptsize $\pm$ 5.4 & 33.8 \scriptsize $\pm$ 5.9 \\
\bottomrule
\end{tabular}
\end{table}

=== 2s-8x8-2p-2f-coop ===


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s-8x8-2p-2f-coop,default,default,ippo,1,0.968750,0.484375,15.421875,1,64
1,2s-8x8-2p-2f-coop,default,default,ippo,2,0.968750,0.484375,16.546875,1,64
2,2s-8x8-2p-2f-coop,default,default,ippo,31,0.984375,0.492188,15.296875,1,64
3,2s-8x8-2p-2f-coop,default,default,ippo,32,0.984375,0.496094,15.875000,1,64
4,2s-8x8-2p-2f-coop,default,default,ipci,1,0.968750,0.484375,17.093750,1,64
5,2s-8x8-2p-2f-coop,default,default,ipci,2,0.953125,0.476563,18.453125,1,64
6,2s-8x8-2p-2f-coop,default,default,ipci,31,0.984375,0.492188,16.765625,1,64
7,2s-8x8-2p-2f-coop,default,default,ipci,32,0.984375,0.492188,17.140625,1,64
8,2s-8x8-2p-2f-coop,default,default,ipci,42,1.000000,0.500000,15.515625,1,64
9,2s-8x8-2p-2f-coop,default,default,mappo,1,0.984375,0.492188,15.015625,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s-8x8-2p-2f-coop,default,default,ippo,0.976562,0.009021,0.489258,0.005859,15.785156,0.565296,4,4,64
1,2s-8x8-2p-2f-coop,default,default,ipci,0.978125,0.017815,0.489063,0.008908,16.993750,1.048599,5,5,64
2,2s-8x8-2p-2f-coop,default,default,mappo,0.978125,0.017815,0.489063,0.008908,14.993750,1.043699,5,5,64
3,2s-8x8-2p-2f-coop,default,default,mpci,0.971875,0.013073,0.487500,0.007514,16.556250,0.647693,5,5,64


\begin{table}[t]
\centering
\caption{Deterministic LBF success rate on 2s-8x8-2p-2f-coop using final checkpoints. Rows are grouped by training constraint. Mean $\pm$ std are computed over seeds, not episodes.}}
\label{tab:lbf_2s-8x8-2p-2f-coop_final}
\begin{tabular}{lcccc}
\toprule
Constraint & ippo & ipci & mappo & mpci \\
\midrule
default & 97.7 \scriptsize $\pm$ 0.9 & 97.8 \scriptsize $\pm$ 1.8 & 97.8 \scriptsize $\pm$ 1.8 & 97.2 \scriptsize $\pm$ 1.3 \\
\bottomrule
\end{tabular}
\end{table}


In [9]:
for env_id, result in ENV_RESULTS.items():
    run_df = result["run_results"].copy()
    seed_summary = result["seed_summary"].copy()
    model_summary = result["model_summary"].copy()

    eval_seed_sets = (
        run_df.groupby(["model_label", "train_seed"])["eval_seed"]
        .agg(lambda values: tuple(sorted(set(int(v) for v in values))))
        .reset_index(name="eval_seed_set")
    )
    assert (eval_seed_sets.groupby("model_label")["eval_seed_set"].nunique() == 1).all()
    assert not (run_df["train_seed"] == run_df["eval_seed"]).any()
    assert (seed_summary["n_runs"] == 1).all() if DUPLICATE_POLICY != "all" else True
    expected_std = seed_summary.groupby("model_label")["eval_success_rate"].std().fillna(0.0)
    observed_std = model_summary.set_index("model_label")["std_success_rate"].fillna(0.0)
    assert (expected_std - observed_std).abs().max() < 1.0e-12

print("Protocol assertions passed.")

Protocol assertions passed.


In [10]:
combined_seed_summary = pd.concat(
    [result["seed_summary"] for result in ENV_RESULTS.values()],
    ignore_index=True,
)
combined_model_summary = pd.concat(
    [result["model_summary"] for result in ENV_RESULTS.values()],
    ignore_index=True,
)
display(combined_seed_summary)
display(combined_model_summary)

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s-10x10-3p-3f,default,default,ippo,1,0.796875,0.297156,34.281250,1,64
1,2s-10x10-3p-3f,default,default,ippo,2,0.859375,0.297115,29.796875,1,64
2,2s-10x10-3p-3f,default,default,ippo,31,0.906250,0.316541,28.234375,1,64
3,2s-10x10-3p-3f,default,default,ippo,32,0.828125,0.293221,28.968750,1,64
4,2s-10x10-3p-3f,default,default,ippo,42,0.750000,0.278814,31.328125,1,64
...,...,...,...,...,...,...,...,...,...,...
74,2s-8x8-2p-2f-coop,default,default,mpci,1,0.968750,0.488281,16.406250,1,64
75,2s-8x8-2p-2f-coop,default,default,mpci,2,0.953125,0.476563,16.312500,1,64
76,2s-8x8-2p-2f-coop,default,default,mpci,31,0.984375,0.496094,15.859375,1,64
77,2s-8x8-2p-2f-coop,default,default,mpci,32,0.984375,0.492188,17.609375,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s-10x10-3p-3f,default,default,ippo,0.828125,0.059498,0.296570,0.013472,30.521875,2.395486,5,5,64
1,2s-10x10-3p-3f,default,default,ipci,0.806250,0.075422,0.292322,0.013144,30.409375,1.606695,5,5,64
2,2s-10x10-3p-3f,default,default,mappo,0.746875,0.077655,0.275414,0.020993,31.200000,1.576593,5,5,64
3,2s-10x10-3p-3f,default,default,mpci,0.762500,0.053446,0.280841,0.012777,30.946875,1.063453,5,5,64
4,2s-15x15-4p-3f,default,default,ippo,0.600000,0.154916,0.197769,0.015080,38.775000,2.980868,5,5,64
5,2s-15x15-4p-3f,default,default,ipci,0.671875,0.057410,0.204866,0.003500,36.350000,1.632480,5,5,64
6,2s-15x15-4p-3f,default,default,mappo,0.653125,0.053446,0.205228,0.006153,36.887500,0.618462,5,5,64
7,2s-15x15-4p-3f,default,default,mpci,0.671875,0.067206,0.206828,0.009449,35.868750,1.281183,5,5,64
8,2s-15x15-4p-5f,default,default,ippo,0.312500,0.081938,0.164819,0.008774,45.521875,1.129310,5,5,64
9,2s-15x15-4p-5f,default,default,ipci,0.365625,0.046351,0.164252,0.007342,44.178125,0.925878,5,5,64
